# Full MVTec AD benchmark — PatchCore (WideResNet50, 1% coreset)
Runs training-free **PatchCore** across **all 15 MVTec AD categories** on a T4 GPU,
loading data from a public Hugging Face mirror. Prints per-category and mean
image / pixel AUROC. **Runtime -> T4 GPU**, then **Run all**.

In [ ]:
# 1) GPU + datasets
import subprocess, sys, torch
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=False)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2) PatchCore over all 15 categories (WRN50, layer2+3, 1% coreset, 224px)
import torch, torch.nn.functional as F, numpy as np, time
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import roc_auc_score
from datasets import load_dataset

torch.manual_seed(42); np.random.seed(42)
dev = "cuda" if torch.cuda.is_available() else "cpu"
REPO = "TheoM55/mvtec_all_objects_split"
SIZE = 224
CATS = ["bottle","cable","capsule","carpet","grid","hazelnut","leather","metal_nut",
        "pill","screw","tile","toothbrush","transistor","wood","zipper"]
mean, std = (0.485,0.456,0.406), (0.229,0.224,0.225)
itf = transforms.Compose([transforms.Resize(256, InterpolationMode.BILINEAR),
                          transforms.CenterCrop(SIZE), transforms.ToTensor(),
                          transforms.Normalize(mean, std)])
mtf = transforms.Compose([transforms.Resize(256, InterpolationMode.NEAREST),
                          transforms.CenterCrop(SIZE), transforms.ToTensor()])

net = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(dev).eval()
for p in net.parameters(): p.requires_grad_(False)
feat = {}
net.layer2.register_forward_hook(lambda m,i,o: feat.__setitem__("l2", o))
net.layer3.register_forward_hook(lambda m,i,o: feat.__setitem__("l3", o))

@torch.no_grad()
def patch_feats(x):
    feat.clear(); net(x.to(dev))
    hw = feat["l2"].shape[-2:]; parts = []
    for k in ["l2","l3"]:
        f = F.avg_pool2d(feat[k], 3, 1, 1)
        parts.append(F.interpolate(f, size=hw, mode="bilinear", align_corners=False))
    e = torch.cat(parts, 1); b,c,h,w = e.shape
    return e.permute(0,2,3,1).reshape(b,h,w,c), (h,w)

def gauss(maps, sigma=4.0):
    r = max(int(4*sigma+0.5),1); xx = torch.arange(-r, r+1.0, device=maps.device)
    k = torch.exp(-0.5*(xx/sigma)**2); k /= k.sum(); m = maps.unsqueeze(1)
    m = F.conv2d(F.pad(m,(r,r,0,0),mode="reflect"), k.view(1,1,1,-1))
    m = F.conv2d(F.pad(m,(0,0,r,r),mode="reflect"), k.view(1,1,-1,1))
    return m.squeeze(1)

def kcenter(Fm, ratio=0.01, pdim=128, seed=42):
    n = Fm.shape[0]; k = max(1, int(n*ratio))
    g = torch.Generator(device="cpu").manual_seed(seed)
    proj = (torch.randn(Fm.shape[1], pdim, generator=g)/pdim**0.5).to(Fm.device, Fm.dtype)
    P = Fm @ proj; s = int(torch.randint(0,n,(1,),generator=g))
    sel=[s]; md_ = torch.cdist(P, P[s:s+1]).squeeze(1)
    for _ in range(1,k):
        j = int(md_.argmax()); sel.append(j)
        md_ = torch.minimum(md_, torch.cdist(P, P[j:j+1]).squeeze(1))
    return torch.tensor(sel, device=Fm.device)

def to_tensors(split):
    imgs, labels, masks = [], [], []
    for ex in split:
        lab = int(ex["label"]); labels.append(lab)
        imgs.append(itf(ex["image_path"].convert("RGB")))
        m = ex.get("mask_path")
        if lab == 0 or m is None:
            masks.append(torch.zeros(1, SIZE, SIZE))
        else:
            masks.append((mtf(m.convert("L")) > 0.5).float())
    return torch.stack(imgs), torch.tensor(labels), torch.stack(masks)

@torch.no_grad()
def run_category(cat):
    tr = load_dataset(REPO, split=f"{cat}.train")
    te = load_dataset(REPO, split=f"{cat}.test")
    Xtr,_,_ = to_tensors(tr); Xte,yte,Mte = to_tensors(te)
    # build bank
    bank = []
    for i in range(0, len(Xtr), 16):
        pf, hw = patch_feats(Xtr[i:i+16]); bank.append(pf.reshape(-1, pf.shape[-1]))
    bank = torch.cat(bank, 0); H,W = hw
    mem = bank[kcenter(bank, 0.01)]
    # eval
    def nn_min(q, chunk=8192):
        return torch.cat([torch.cdist(q[s:s+chunk], mem).min(1).values
                          for s in range(0,q.shape[0],chunk)], 0)
    scores=[]; maps=[]
    for i in range(0, len(Xte), 16):
        pf,_ = patch_feats(Xte[i:i+16]); b = pf.shape[0]; q = pf.reshape(-1, pf.shape[-1])
        d = nn_min(q).reshape(b,H,W); qf = q.reshape(b,H*W,-1); ps = d.reshape(b,H*W)
        for j in range(b):
            p = int(ps[j].argmax()); xs = qf[j,p:p+1]
            im = int(torch.cdist(xs, mem).argmin()); ms = mem[im:im+1]
            nb = mem[torch.topk(torch.cdist(ms, mem).squeeze(0), 9, largest=False).indices]
            dxn = torch.cdist(xs, nb).squeeze(0)
            scores.append(float((1 - torch.softmax(dxn,0)[0]) * ps[j,p]))
        sm = F.interpolate(d.unsqueeze(1), size=(SIZE,SIZE), mode="bilinear", align_corners=False).squeeze(1)
        maps.append(gauss(sm).cpu())
    MP = torch.cat(maps,0).numpy()
    ia = roc_auc_score(yte.numpy(), np.array(scores))
    pa = roc_auc_score((Mte.squeeze(1).numpy().reshape(-1) > 0.5).astype(int), MP.reshape(-1))
    del bank, mem; torch.cuda.empty_cache()
    return ia, pa

rows = []
print(f"{'category':12s}  image  pixel")
for cat in CATS:
    t = time.time()
    try:
        ia, pa = run_category(cat)
        rows.append((cat, ia, pa))
        print(f"{cat:12s}  {ia:.3f}  {pa:.3f}   ({time.time()-t:.0f}s)")
    except Exception as e:
        print(f"{cat:12s}  FAILED: {e}")

mi = np.mean([r[1] for r in rows]); mp = np.mean([r[2] for r in rows])
print("\n===CSV_START===")
print("category,image_auroc,pixel_auroc")
for c,ia,pa in rows: print(f"{c},{ia:.4f},{pa:.4f}")
print(f"MEAN,{mi:.4f},{mp:.4f}")
print("===CSV_END===")